# GuardSLM: LMS Action Authorization Benchmark (Research Snapshots)
### Context-Aware Permission Gatekeeping for Browser-Based AI Agents

This notebook provides the evaluation pipeline for the canonical **GuardSLM LMS Research Snapshots** (`research_snapshots/`).

All models evaluate on authentic **Playwright accessibility tree snapshots** captured from the live GuardSLM LMS research environment. Each model is evaluated on **20 snapshot cases (10 complete counterfactual matched pairs)** using its authentic native input and output format with zero leakage of gold labels.

### Model Matrix

| Model Name | Model Type | Parameters | Evaluation Modality | Subset Limit | Notes |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **PolicyGuard-4B** | Trajectory Safety Guard | 4.0B | Accessibility Tree & Trajectory | 20 Cases (10 Pairs) | Native web agent trajectory format (`Step 1: Request, Step 2: Role, Step 3: State Context/A11y Tree, Step 4: Action` → `violation` / `no_violation`). |
| **DynaGuard-1.7B** | Policy Compliance Gatekeeper | 1.7B | Accessibility Tree & Context | 20 Cases (10 Pairs) | Native rule compliance format (`USER ROLE, REQUEST, CURRENT CONTEXT/A11Y, POLICY, PROPOSED ACTION` → `ALLOW` / `BLOCK` / `ASK`). |
| **WebGuard-3B** | Action Risk Guard | 3.0B | Accessibility Tree Risk | 20 Cases (10 Pairs) | Native consequence risk format (`{"risky_level": "SAFE" | "LOW" | "HIGH"}`). |
| **Qwen3-VL 2B Instruct** | Multimodal VLM | 2.0B | Text-Only | 20 Cases (10 Pairs) | Text-only authorization evaluation without screenshots. |
| **Mock Benchmark Guard** | Pipeline Simulation | 0 | Text-Only | 20 Cases (10 Pairs) | Fast CPU test harness for verification. |

---


### Step 1 — Environment Setup & Path Configuration
Clone the repository (if on Google Colab) or configure workspace working directory.

In [ ]:
import os, sys

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo_dir = "/content/GuardSLM"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/AkarshiAaryan/GuardSLM.git {repo_dir}
    os.chdir(repo_dir)
    print("Running inside Google Colab! Changed working directory to:", os.getcwd())
else:
    print("Running locally in working directory:", os.getcwd())

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


### Step 2 — Install Dependencies
Install PyTorch, Hugging Face Transformers, Accelerate, BitsAndBytes, and project requirements.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate
!pip install -q -r requirements.txt
print("Dependencies successfully installed!")


### Step 3 — Hardware & Accelerator Check
Verify CUDA GPU availability, device name, and total VRAM.

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA Accelerator Available: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("Running in CPU mode. GPU acceleration is strongly recommended for model inference.")


### Step 4 — Inspect Benchmark Dataset & Verification
Load and inspect the canonical snapshot dataset from `research_snapshots`.

In [ ]:
from src.benchmark.data.loader import BenchmarkLoader

loader = BenchmarkLoader("research_snapshots")
cases = loader.get_model_inputs()
golds = loader.get_gold_annotations()
pairs = loader.get_pair_registry()

print(f"Dataset Name: {loader.dataset_name}")
print(f"Total Test Cases: {len(cases)}")
print(f"Total Private Gold Annotations: {len(golds)}")
print(f"Total Matched Pairs: {len(pairs)}")

sample = cases[0]
print(f"Sample Case ID: {sample.case_id}")
print(f"Sample User Request: {sample.user_request}")
print(f"Sample Effective Role: {sample.authenticated_role}")
print(f"Sample Target Resource: {sample.target_resource}")
print(f"Sample Accessibility Tree Length: {len(sample.target_a11y_tree)} chars")


### Step 5 — Fast Pipeline Smoke Test (Mock Guard on 4 Cases)
Execute the complete pipeline on a small 4-case subset to verify execution, parsing, and report generation.

In [ ]:
from src.benchmark.runner.experiment_runner import run_benchmark_experiment

smoke_results = run_benchmark_experiment(
    dataset_path="research_snapshots",
    model_name="mock_guard",
    model_config={"mode": "perfect"},
    strategy="text_only",
    limit=4,
    output_dir="data/benchmark_results/smoke_test"
)
print("Smoke test completed successfully!")
print("Accuracy:", smoke_results["text_only"]["decision_performance"]["accuracy"])


### Step 6A — Model 1: PolicyGuard-4B (Trajectory Safety Guard — 20 Cases)
Evaluate **PolicyGuard-4B** (`Rakancorle1/PolicyGuard-4B`) using its native web agent trajectory format (`Step 1: Request, Step 2: Role, Step 3: State Context/A11y Tree, Step 4: Action` → `violation` / `no_violation`).
Evaluated on the 20 snapshot test cases (10 complete matched pairs: `limit=20`).

In [ ]:
policyguard_results = run_benchmark_experiment(
    dataset_path="research_snapshots",
    model_name="policyguard",
    model_config={"checkpoint": "Rakancorle1/PolicyGuard-4B", "load_in_4bit": False},
    strategy="text_only",
    limit=20,
    resume=True
)
print("PolicyGuard Decision Accuracy:", policyguard_results["text_only"]["decision_performance"]["accuracy"])
print("PolicyGuard Pair Sensitivity:", policyguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 6B — Model 2: DynaGuard-1.7B (Policy Compliance Gatekeeper — 20 Cases)
Evaluate **DynaGuard-1.7B** (`tomg-group-umd/DynaGuard-1.7B`) using its native rule-following gatekeeper format (`USER ROLE, USER REQUEST, CURRENT CONTEXT, RELEVANT POLICY, PROPOSED ACTION` → `ALLOW` / `BLOCK` / `ASK`).
Evaluated on the 20 snapshot test cases (10 complete matched pairs: `limit=20`).

In [ ]:
dynaguard_results = run_benchmark_experiment(
    dataset_path="research_snapshots",
    model_name="dynaguard",
    model_config={"checkpoint": "tomg-group-umd/DynaGuard-1.7B", "load_in_4bit": False},
    strategy="text_only",
    limit=20,
    resume=True
)
print("DynaGuard Decision Accuracy:", dynaguard_results["text_only"]["decision_performance"]["accuracy"])
print("DynaGuard Pair Sensitivity:", dynaguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 6C — Model 3: WebGuard-3B (Accessibility Tree Action Guard — 20 Cases)
Evaluate **WebGuard-3B** (`Qwen/Qwen2.5-3B-Instruct` base or with LoRA adapter) using its native accessibility-tree consequence format (`{"risky_level": "SAFE" | "LOW" | "HIGH"}`).
Evaluated on the 20 snapshot test cases (10 complete matched pairs: `limit=20`).

In [ ]:
webguard_results = run_benchmark_experiment(
    dataset_path="research_snapshots",
    model_name="webguard",
    model_config={"checkpoint": "Qwen/Qwen2.5-3B-Instruct", "adapter_path": "models/webguard_qwen_lora", "augmented": True, "load_in_4bit": False},
    strategy="text_only",
    limit=20,
    resume=True
)
print("WebGuard Decision Accuracy:", webguard_results["text_only"]["decision_performance"]["accuracy"])
print("WebGuard Pair Sensitivity:", webguard_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 7 — Model 4: Qwen3-VL 2B (Text-Only Authorization Evaluation — 20 Cases)
Evaluate **Qwen3-VL 2B** under text-only authorization evaluation on the snapshot dataset.
Evaluated on the 20 snapshot test cases (10 complete matched pairs: `limit=20`).

In [ ]:
qwen3_vl_results = run_benchmark_experiment(
    dataset_path="research_snapshots",
    model_name="qwen3_vl_2b",
    model_config={"checkpoint": "Qwen/Qwen3-VL-2B-Instruct", "is_vlm": True, "load_in_4bit": False},
    strategy="text_only",
    limit=20,
    resume=True
)
print("Qwen3-VL Decision Accuracy:", qwen3_vl_results["text_only"]["decision_performance"]["accuracy"])
print("Qwen3-VL Pair Sensitivity:", qwen3_vl_results["text_only"]["matched_pair_performance"]["authorization_flip_accuracy"])


### Step 8 — Consolidated Multi-Model Leaderboard & Comparative Reports
Synthesize side-by-side comparative performance tables across all evaluated models, followed by each model's full individual breakdown.

In [ ]:
import glob
import os
from IPython.display import display, Markdown
from src.benchmark.evaluation.evaluator import generate_consolidated_summary_table

# 1. Display Cross-Model Comparative Leaderboard
leaderboard_md = generate_consolidated_summary_table("data/benchmark_results")
display(Markdown(leaderboard_md))

# 2. Discover and Display Every Evaluated Model's Report
report_files = sorted(glob.glob("data/benchmark_results/*/report_*.md"))
print(f"\nDiscovered {len(report_files)} individual benchmark reports:")
for r in report_files:
    print(" - ", r)

for r in report_files:
    exp_dir = os.path.basename(os.path.dirname(r))
    display(Markdown(f"\n---\n## 📋 Detailed Report: `{exp_dir}`\n"))
    with open(r, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
